# Backtesting

Imports

In [91]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm

### Step 1: Load cointegration and price data

In [92]:
data_folder_path = Path("data")
print(data_folder_path.exists())

True


In [93]:
bh_corrected_df = pd.read_parquet(data_folder_path / "bh_corrected_results.parquet")
bh_sig_results_df = bh_corrected_df[bh_corrected_df["bh_reject"]]
sig_results_df = pd.read_parquet(data_folder_path / "sig_results.parquet")

In [94]:
print(sig_results_df.head())

                                 correlation   coint_t   p_value  \
formation_end ticker_0 ticker_1                                    
2016-01-01    STT      GS           0.753596 -3.362022  0.046747   
                       MS           0.739525 -4.778833  0.000411   
                       LNC          0.776851 -3.481309  0.034089   
              COP      EOG          0.781141 -3.454777  0.036620   
              KIM      O            0.762781 -4.339056  0.002224   

                                 critical_1pct  critical_5pct  critical_10pct  \
formation_end ticker_0 ticker_1                                                 
2016-01-01    STT      GS            -3.918346      -3.348304       -3.052893   
                       MS            -3.918346      -3.348304       -3.052893   
                       LNC           -3.918346      -3.348304       -3.052893   
              COP      EOG           -3.918346      -3.348304       -3.052893   
              KIM      O             

In [95]:
print(bh_corrected_df.head())

                                 correlation   coint_t   p_value  \
formation_end ticker_0 ticker_1                                    
2016-01-01    SWKS     AVGO         0.705393 -0.019827  0.985429   
              BAC      STT          0.730833 -2.431513  0.310079   
                       HBAN         0.767496 -2.654967  0.216192   
                       PFG          0.710855 -2.270825  0.388197   
                       TFC          0.756456 -2.563165  0.251944   

                                 critical_1pct  critical_5pct  critical_10pct  \
formation_end ticker_0 ticker_1                                                 
2016-01-01    SWKS     AVGO          -3.918346      -3.348304       -3.052893   
              BAC      STT           -3.918346      -3.348304       -3.052893   
                       HBAN          -3.918346      -3.348304       -3.052893   
                       PFG           -3.918346      -3.348304       -3.052893   
                       TFC           

In [96]:
historical_ohlcv_df = pd.read_parquet(data_folder_path / "historical_ohlcv_df.parquet")

In [97]:
print(historical_ohlcv_df.head())

Ticker      PCL                        LLL                           ...  \
Price      Open High Low Close Volume Open High Low Close Adj Close  ...   
Date                                                                 ...   
2014-01-02  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-03  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-06  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-07  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-08  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   

Ticker           ACGL                                                 ROST  \
Price            Open       High        Low      Close   Volume       Open   
Date                                                                         
2014-01-02  18.783414  18.875336  18.162162  18.184349  2592900  32.832875   
2014-01-03  18.152653  18.219215  17.753275  17.835686  3620400  32.819630   
2

### Step 2: Estimate relationship parameters for all pairs

In [98]:
close_prices = historical_ohlcv_df.xs("Close", level="Price", axis=1)
print(close_prices.head())

Ticker      PCL  LLL  GMCR  MRO  ETFC  DO  RTN  GGP  NFX  LEG  ...        XEL  \
Date                                                           ...              
2014-01-02  NaN  NaN   NaN  NaN   NaN NaN  NaN  NaN  NaN  NaN  ...  18.308685   
2014-01-03  NaN  NaN   NaN  NaN   NaN NaN  NaN  NaN  NaN  NaN  ...  18.302029   
2014-01-06  NaN  NaN   NaN  NaN   NaN NaN  NaN  NaN  NaN  NaN  ...  18.202192   
2014-01-07  NaN  NaN   NaN  NaN   NaN NaN  NaN  NaN  NaN  NaN  ...  18.514999   
2014-01-08  NaN  NaN   NaN  NaN   NaN NaN  NaN  NaN  NaN  NaN  ...  18.441788   

Ticker            APA         CMI        AWK          D        WAB       NDAQ  \
Date                                                                            
2014-01-02  63.316505   99.743729  31.939985  37.304127  68.494621  10.673878   
2014-01-03  63.360950  100.175621  31.847795  37.263054  68.934059  10.831811   
2014-01-06  63.931301   99.858910  31.724890  37.268929  68.148705  10.859035   
2014-01-07  65.109055  100.

Create a function to estimate params for a pair

In [99]:
def estimate_params(ticker_0, ticker_1, formation_start, formation_end):
    pair_prices = close_prices.loc[
        formation_start:(formation_end - pd.DateOffset(days=1)),
        [ticker_0, ticker_1]
    ].dropna()

    y = pair_prices[ticker_0]
    x = pair_prices[ticker_1]

    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params[ticker_1]

    spread = y - (alpha + beta * x)

    return alpha, beta, spread.mean(), spread.std()

In [100]:
print(estimate_params("AAPL", "MSFT", pd.Timestamp("2020-01-01"), pd.Timestamp("2020-12-31")))

(np.float64(-70.92462209671135), np.float64(0.8908157356582398), np.float64(3.620384415539558e-14), np.float64(7.600240592432756))


Create a function to estimate params for all pairs in a window and store in a dataframe

In [101]:
def estimate_params_for_window(pairs, formation_start, formation_end):
    results = []

    for ticker_0, ticker_1 in pairs:
        alpha, beta, spread_mean, spread_std = estimate_params(ticker_0, ticker_1, formation_start, formation_end)
        results.append({
            "formation_end": formation_end,
            "ticker_0": ticker_0,
            "ticker_1": ticker_1,
            "alpha": alpha,
            "beta": beta,
            "spread_mean": spread_mean,
            "spread_std": spread_std
        })
    return pd.DataFrame(results).set_index(["formation_end", "ticker_0", "ticker_1"])

Create a function to estimate params for all windows in a dataframe of cointegration data. We will call this function twice: once on the raw p-value threshold data and once on the BH-corrected data.

In [102]:
def estimate_params_for_all_windows(coint_df, window_size=504):
    results = []

    formation_ends = coint_df.index.get_level_values("formation_end").unique()

    for formation_end in formation_ends:
        end_idx = close_prices.index.searchsorted(formation_end, side="left") - 1
        formation_start = close_prices.index[end_idx - (window_size - 1)]

        pairs = coint_df.loc[formation_end].index.tolist()

        window_params_df = estimate_params_for_window(
            pairs,
            formation_start,
            formation_end
        )

        results.append(window_params_df)

    return pd.concat(results, ignore_index=False)

Estimate params for both sig_results_df and bh_corrected_df

In [103]:
sig_window_params_df = estimate_params_for_all_windows(sig_results_df)
bh_window_params_df = estimate_params_for_all_windows(bh_sig_results_df)

In [104]:
sig_window_params_df

alpha      beta   spread_mean  spread_std
formation_end ticker_0 ticker_1                                               
2016-01-01    STT      GS        15.529632  0.254871 -2.083131e-13    1.591032
                       MS        15.050681  1.496055  1.448154e-13    1.330992
                       LNC        9.530156  1.223589 -9.741766e-15    1.630143
              COP      EOG       -8.118955  0.813194 -9.637441e-14    2.007876
              KIM      O          0.943564  0.495747 -7.806806e-15    0.274850
...                                    ...       ...           ...         ...
2026-01-01    MET      MTB       36.637323  0.220793  5.936697e-14    2.304742
              CFG      AXP       -0.003349  0.147049  6.084727e-14    2.151317
              AXP      MS        68.444542  1.757341 -4.849736e-15   11.097357
              COF      WFC       -7.395712  2.698647  5.188090e-15    7.228975
              GIS      CAG       14.885463  1.787903 -4.116636e-15    1.451645

[14542 rows x 4 columns]

In [105]:
bh_window_params_df

alpha      beta   spread_mean  spread_std
formation_end ticker_0 ticker_1                                               
2016-01-01    XEL      CMS        3.514137  0.847090 -8.742918e-14    0.228813
2016-02-01    XEL      CMS        3.464457  0.849943  5.378414e-14    0.236661
2016-03-01    STT      MS        10.544865  1.664574  1.141944e-14    1.125561
              XEL      CMS        3.481570  0.849850  2.403721e-15    0.234388
2016-04-01    STT      MS        11.365381  1.633517 -4.821540e-15    1.122996
...                                    ...       ...           ...         ...
2025-11-01    PPL      NI         8.675948  0.647139  1.404873e-14    0.611208
              CPT      ESS        3.367895  0.392093  1.248243e-13    2.017917
              RF       KEY       -1.634339  1.484203  3.118493e-14    0.445216
2025-12-01    CPT      ESS        2.656223  0.394576 -1.356798e-13    2.043802
              RF       KEY       -2.206337  1.521070  1.239220e-14    0.419373

[406 rows x 4 columns]

### Step 3: Simulate trading using formation window data

Calculate z-score from 2 prices

In [106]:
def calculate_z_score(price_0, price_1, alpha, beta, spread_mean, spread_std):
    spread = price_0 - (alpha + beta * price_1) # y is ticker_0, x is ticker_1 according to our earlier code
    return (spread - spread_mean) / spread_std

In [107]:
TRADE_THRESH = 2 # only trade if absolute value of z-score is > 2
SELL_THRESH = 0.5 # we will sell our current portfolio if absolute value of z score < 0.5
TRADE_NOMINAL = 10000 # spend $10k per trade
TRANSACTION_COST_RATE = 0.0005 # 5 bps per trade

Function to open a position

In [108]:
open_positions = {}
trade_history = []

def open_position(ticker_0, ticker_1, formation_end, direction, signal_date, entry_date, entry_z, alpha, beta, q0, q1, entry_price_0, entry_price_1):
    pair = (ticker_0, ticker_1)

    if pair in open_positions:
        raise ValueError(f"Position for pair {pair} is already open.")

    entry_gross_exposure = abs(q0 * entry_price_0) + abs(q1 * entry_price_1)
    entry_cost = TRANSACTION_COST_RATE * entry_gross_exposure

    open_positions[pair] = {
        "formation_end": formation_end,
        "direction": direction,
        "signal_date": signal_date,
        "entry_date": entry_date,
        "entry_z": entry_z,
        "alpha": alpha,
        "beta": beta,
        "q0": q0,
        "q1": q1,
        "entry_price_0": entry_price_0,
        "entry_price_1": entry_price_1,
        "entry_gross_exposure": entry_gross_exposure,
        "entry_cost": entry_cost
    }

Function to close a position

In [109]:
def close_position(ticker_0, ticker_1, exit_signal_date, exit_date, exit_z, exit_price_0, exit_price_1, exit_reason):
    pair = (ticker_0, ticker_1)

    if pair not in open_positions:
        raise ValueError(f"Position for pair {pair} is not open.")

    position = open_positions[pair]

    q0 = position["q0"]
    q1 = position["q1"]

    gross_pnl = q0 * (exit_price_0 - position["entry_price_0"]) + q1 * (exit_price_1 - position["entry_price_1"])

    exit_gross_exposure = abs(q0 * exit_price_0) + abs(q1 * exit_price_1)
    exit_cost = TRANSACTION_COST_RATE * exit_gross_exposure
    total_cost = position["entry_cost"] + exit_cost
    net_pnl = gross_pnl - total_cost
    trade_return = net_pnl / position["entry_gross_exposure"]
    holding_days = (exit_date - position["entry_date"]).days

    trade_history.append({
        "ticker_0": ticker_0,
        "ticker_1": ticker_1,
        "formation_end": position["formation_end"],
        "direction": position["direction"],
        "entry_signal_date": position["signal_date"],
        "entry_date": position["entry_date"],
        "exit_signal_date": exit_signal_date,
        "exit_date": exit_date,
        "entry_z": position["entry_z"],
        "exit_z": exit_z,
        "alpha": position["alpha"],
        "beta": position["beta"],
        "q0": q0,
        "q1": q1,
        "entry_price_0": position["entry_price_0"],
        "entry_price_1": position["entry_price_1"],
        "exit_price_0": exit_price_0,
        "exit_price_1": exit_price_1,
        "entry_gross_exposure": position["entry_gross_exposure"],
        "exit_gross_exposure": exit_gross_exposure,
        "gross_pnl": gross_pnl,
        "entry_cost": position["entry_cost"],
        "exit_cost": exit_cost,
        "total_cost": total_cost,
        "net_pnl": net_pnl,
        "trade_return": trade_return,
        "holding_days": holding_days,
        "exit_reason": exit_reason
    })

    del open_positions[pair]

Quick test to confirm that price data is stored for the first trading day of 2026

In [ ]:
historical_ohlcv_df.index[-1]

Timestamp('2026-01-02 00:00:00')

Function to trade within a window for a given df

In [ ]:
missing_price_events = [] # store any time that price data suddenly becomes NaN for a ticker. These pairs will simply be skipped for now.

In [ ]:
def simulate_trading_window(params_df, formation_end):
    start_date = formation_end
    target_end_date = formation_end + pd.DateOffset(months=1)
    end_date = historical_ohlcv_df.index[historical_ohlcv_df.index.searchsorted(target_end_date)] # get the first trading day of next window

    trading_ohlcv_df = historical_ohlcv_df.loc[
        (historical_ohlcv_df.index >= start_date) &
        (historical_ohlcv_df.index <= end_date)
    ]

    trading_close_df = trading_ohlcv_df.xs('Close', level='Price', axis=1).iloc[:-1] # used for signals
    trading_open_df = trading_ohlcv_df.xs('Open', level='Price', axis=1).iloc[1:] # used for returns

    formation_params = params_df.xs(formation_end, level="formation_end")
    pairs = formation_params.index.tolist()

    #print(trading_close_df)
    for curr_date, open_date in zip(trading_close_df.index, trading_open_df.index):
        daily_close_prices = trading_close_df.loc[curr_date]
        next_open_prices = trading_open_df.loc[open_date]
        for ticker_0, ticker_1 in pairs:
            pair = (ticker_0, ticker_1)
            # get prices for each ticker
            close_price_0 = daily_close_prices[ticker_0]
            close_price_1 = daily_close_prices[ticker_1]
            entry_price_0 = next_open_prices[ticker_0]
            entry_price_1 = next_open_prices[ticker_1]

            if pd.isna(close_price_0) or pd.isna(close_price_1) or pd.isna(entry_price_0) or pd.isna(entry_price_1):
                missing_price_events.append((pair, curr_date, open_date))
                continue

            # get params & z-score
            curr_pair_params = formation_params.loc[pair] # extracts a Series since pairs are unique
            alpha = curr_pair_params["alpha"]
            beta = curr_pair_params["beta"]
            spread_mean = curr_pair_params["spread_mean"]
            spread_std = curr_pair_params["spread_std"]
            z_score = calculate_z_score(close_price_0, close_price_1, alpha, beta, spread_mean, spread_std)

            if (curr_date == trading_close_df.index[-1]):
                if pair in open_positions:
                    close_position(ticker_0, ticker_1, curr_date, open_date, z_score, next_open_prices[ticker_0], next_open_prices[ticker_1], "window_end")
                continue

            if pair in open_positions:
                if (z_score * open_positions[pair]["direction"] <= 0):
                    close_position(ticker_0, ticker_1, curr_date, open_date, z_score, next_open_prices[ticker_0], next_open_prices[ticker_1], "mean_reversion")

                continue


            if (abs(z_score) >= 2):
                scale = TRADE_NOMINAL / (entry_price_0 + abs(beta) * entry_price_1)

                direction = z_score / abs(z_score)

                q0 = -direction * scale # shares of ticker_0
                q1 = direction * beta * scale # shares of ticker_1
                # execute trade for this pair
                open_position(ticker_0, ticker_1, formation_end, direction, curr_date, open_date, z_score, alpha, beta, q0, q1, entry_price_0, entry_price_1)

1711.052123163587 1861.8125154989325 370000.0 5 37
0.4624465197739425
3.7236250309978653


### Step 4: Simulate all trading windows over 10-year period

In [ ]:
def simulate_all_windows(params_df):
    formation_ends = params_df.index.levels[0]
    if (formation_ends[-1] >= pd.Timestamp("2026-01-01")): # no data for January 2026 or later, so exclude these dates
        formation_ends = formation_ends[:-1]
    for formation_end in formation_ends:
        simulate_trading_window(params_df, formation_end)

In [ ]:
trade_history.clear()
simulate_all_windows(sig_window_params_df)
trade_history

[{'ticker_0': 'TFC', 'ticker_1': 'ZION', 'formation_end': Timestamp('2017-11-01 00:00:00'), 'direction': np.float64(1.0), 'entry_signal_date': Timestamp('2017-11-01 00:00:00'), 'entry_date': Timestamp('2017-11-02 00:00:00'), 'exit_signal_date': Timestamp('2017-11-14 00:00:00'), 'exit_date': Timestamp('2017-11-15 00:00:00'), 'entry_z': np.float64(2.6806553359067085), 'exit_z': np.float64(-0.5230126433908037), 'alpha': np.float64(11.661770042038992), 'beta': np.float64(0.5935059150884472), 'q0': np.float64(-183.26882002948201), 'q1': np.float64(108.77112873877766), 'entry_price_0': np.float64(33.85455465467915), 'entry_price_1': np.float64(34.89451437916556), 'exit_price_0': np.float64(32.01509775550907), 'exit_price_1': np.float64(34.88132453983918), 'entry_gross_exposure': np.float64(10000.0), 'exit_gross_exposure': np.float64(9661.450230882598), 'gross_pnl': np.float64(335.68042169457715), 'entry_cost': np.float64(5.0), 'exit_cost': np.float64(4.830725115441299), 'total_cost': np.floa

In [ ]:
trade_history.clear()

    # print(pairs)
simulate_trading_window(sig_window_params_df, pd.Timestamp("2017-11-01"))
total_gross_exposure = 0
total_net_pnl = 0
total_net_mean_reversion_pnl = 0
num_mean_reverted = 0
total_returns = 0
total_mean_reversion_returns = 0
for entry in trade_history:
    total_gross_exposure += entry["entry_gross_exposure"]
    total_net_pnl += entry["net_pnl"]
    total_returns += entry["trade_return"]
    if entry["exit_reason"] == "mean_reversion":
        num_mean_reverted += 1
        total_net_mean_reversion_pnl += entry["net_pnl"]
        total_mean_reversion_returns += entry["trade_return"]
print(total_net_pnl, total_net_mean_reversion_pnl, total_gross_exposure, num_mean_reverted, len(trade_history))
print(total_returns / len(trade_history) * 100)
if (num_mean_reverted > 0):
    print(total_mean_reversion_returns / num_mean_reverted * 100)